# EnergyFlow.jl — `emd!` Benchmark

Benchmarks the `emd!` (and `emd_ns64!`) function across different problem sizes,
comparing serial vs parallel cost-matrix fill, and measuring allocations.

In [ ]:
using DelimitedFiles, BenchmarkTools

push!(LOAD_PATH, joinpath(@__DIR__, "..", "src"))
using EnergyFlow

println("Julia version: ", VERSION)
println("Threads:       ", Threads.nthreads())

BenchmarkTools.DEFAULT_PARAMETERS.seconds = 5.0

# Path to pre-generated data (weights & 2D coordinates)
datadir = joinpath(@__DIR__, "..", "..", "data")
println("Data dir:      ", datadir)

## 1. Helper: load events from CSV

In [ ]:
"""
Load a pair of EnergyFlow-format event matrices for a given particle count `n`.
Returns `(ev0, ev1)` where each is M×3 (weight, y, φ).
"""
function load_events(n::Int)
    w0 = vec(readdlm(joinpath(datadir, "w0_n$(n).csv"), ',', Float64))
    w1 = vec(readdlm(joinpath(datadir, "w1_n$(n).csv"), ',', Float64))
    c0 = readdlm(joinpath(datadir, "c0_n$(n).csv"), ',', Float64)
    c1 = readdlm(joinpath(datadir, "c1_n$(n).csv"), ',', Float64)
    ev0 = hcat(w0, c0)
    ev1 = hcat(w1, c1)
    return ev0, ev1
end

# Available problem sizes
sizes = [2, 10, 50, 100, 500, 1000, 2000, 5000]
println("Problem sizes: ", sizes)

## 2. Single-pair `emd!` — serial vs parallel cost fill

For each problem size we benchmark `emd!` with:
- **Serial** cost fill (`parallel_threshold` set above n²)
- **Parallel** cost fill (`parallel_threshold = 1`)

In [ ]:
results = []

for n in sizes
    ev0, ev1 = load_events(n)

    # --- Serial ---
    ws_ser = EMDWorkspace{Float64}(n, n; beta=1.0, R=1.0, norm=true)
    ws_ser.parallel_threshold = n * n + 1   # force serial
    emd_ns64!(ws_ser, ev0, ev1)             # warmup
    b_ser = @benchmark emd_ns64!($ws_ser, $ev0, $ev1)

    # --- Parallel ---
    ws_par = EMDWorkspace{Float64}(n, n; beta=1.0, R=1.0, norm=true)
    ws_par.parallel_threshold = 1            # force parallel
    emd_ns64!(ws_par, ev0, ev1)             # warmup
    b_par = @benchmark emd_ns64!($ws_par, $ev0, $ev1)

    med_ser = median(b_ser.times) / 1_000   # µs
    med_par = median(b_par.times) / 1_000
    speedup = med_ser / med_par

    push!(results, (;
        n,
        serial_µs  = round(med_ser; digits=2),
        parallel_µs = round(med_par; digits=2),
        speedup    = round(speedup; digits=2),
        allocs_ser = b_ser.allocs,
        allocs_par = b_par.allocs,
        mem_ser_KiB = round(b_ser.memory / 1024; digits=1),
        mem_par_KiB = round(b_par.memory / 1024; digits=1),
    ))

    println("n=$n:  serial=$(round(med_ser; digits=1)) µs  parallel=$(round(med_par; digits=1)) µs  speedup=$(round(speedup; digits=2))x")
end

In [ ]:
# Pretty-print results table
println(rpad("n", 8), rpad("serial(µs)", 14), rpad("parallel(µs)", 15),
        rpad("speedup", 10), rpad("alloc_s", 10), rpad("alloc_p", 10),
        rpad("mem_s(KiB)", 12), "mem_p(KiB)")
println("-"^90)
for r in results
    println(rpad(r.n, 8), rpad(r.serial_µs, 14), rpad(r.parallel_µs, 15),
            rpad(r.speedup, 10), rpad(r.allocs_ser, 10), rpad(r.allocs_par, 10),
            rpad(r.mem_ser_KiB, 12), r.mem_par_KiB)
end

## 3. Backend-dispatched `emd!` benchmark

Verify that calling through the backend orchestrator (`emd!`) adds no overhead
compared to calling the NS backend directly (`emd_ns64!`).

In [ ]:
n = 500
ev0, ev1 = load_events(n)
ws = EMDWorkspace{Float64}(n, n; beta=1.0, R=1.0, norm=true)

# Warmup
emd!(ws, ev0, ev1)
emd_ns64!(ws, ev0, ev1)

b_dispatch = @benchmark emd!($ws, $ev0, $ev1)
b_direct   = @benchmark emd_ns64!($ws, $ev0, $ev1)

println("emd! (dispatched): median = $(round(median(b_dispatch.times)/1000; digits=2)) µs")
println("emd_ns64! (direct): median = $(round(median(b_direct.times)/1000; digits=2)) µs")
println("Overhead: $(round(median(b_dispatch.times) / median(b_direct.times); digits=4))x")

## 4. Allocation check

Verify that `emd_ns64!` with a pre-allocated workspace has minimal allocations
(only from `_unpack_event` creating temporary weight/coord arrays).

In [ ]:
for n in [10, 100, 1000]
    ev0, ev1 = load_events(n)
    ws = EMDWorkspace{Float64}(n, n; beta=1.0, R=1.0, norm=true)
    emd_ns64!(ws, ev0, ev1)  # warmup

    allocs = @allocated emd_ns64!(ws, ev0, ev1)
    println("n=$n: @allocated = $(allocs) bytes ($(round(allocs/1024; digits=1)) KiB)")
end

## 5. Scaling plot data

Collect median times for plotting (serial only) to observe algorithmic scaling.

In [ ]:
ns_vec = Int[]
times_µs = Float64[]

for n in sizes
    ev0, ev1 = load_events(n)
    ws = EMDWorkspace{Float64}(n, n; beta=1.0, R=1.0, norm=true)
    ws.parallel_threshold = n * n + 1  # force serial for clean measurement
    emd_ns64!(ws, ev0, ev1)  # warmup

    b = @benchmark emd_ns64!($ws, $ev0, $ev1)
    push!(ns_vec, n)
    push!(times_µs, median(b.times) / 1_000)
end

println("\nScaling data (serial emd_ns64!):")
println(rpad("n", 8), "median (µs)")
for (n, t) in zip(ns_vec, times_µs)
    println(rpad(n, 8), round(t; digits=2))
end